# M1 - Régression Lasso (Modèle Pur)

**Objectif** : Construire un modèle robuste basé exclusivement sur la régression linéaire (Lasso) pour prédire `SalePrice`.

**Actions** :
- Nettoyage des valeurs manquantes et des erreurs de saisie
- Feature engineering manuel pour enrichir les données
- Validation croisée avec `LassoCV`
- Calcul des métriques Mathématiques (RMSE Log), IT (Temps d'entraînement) et Métier (Écart médian)

## 1. Importation des librairies

**Objectif** : Charger les outils nécessaires à l'analyse et à la modélisation.

**Actions** :
- Import des librairies standards (`pandas`, `numpy`, etc.)
- Import de `Scikit-Learn` (`LassoCV`, `Pipeline`, etc.)
- Import de `time` pour le suivi des performances IT.

In [29]:
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from category_encoders import TargetEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor

## 2. Chargement des données et Nettoyage

**Objectif** : Préparer le dataset brut.

**Actions** :
- Chargement de `train.csv` et `test.csv`.
- Suppression des valeurs extrêmes (Outliers).
- Traitement des erreurs de saisie manifestes.

In [30]:
# 1. CHARGEMENT DES DONNEES
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

# Outliers reconnus dans Ames Housing (selon le document de recherche fournit avec le cas)
train_df = train_df[train_df['GrLivArea'] < 4000].reset_index(drop=True)

# Analyse rapide des valeurs manquantes sur train + test
all_df_temp = pd.concat([train_df.drop('SalePrice', axis=1), test_df], axis=0, ignore_index=True)
missing_counts = all_df_temp.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

print(f'DONNÉES MANQUANTES ({len(missing_counts)} colonnes)')
for col, count in missing_counts.items():
    pct = 100 * count / len(all_df_temp)
    print(f'   {col:20s} : {count:4d} ({pct:5.1f}%)')

# Suppression de lignes uniquement si le NA est considéré comme une erreur de saisie
error_missing_cols = [
    'MSZoning', 'BsmtFullBath', 'Functional', 'BsmtHalfBath', 'Utilities',
    'BsmtFinSF1', 'Exterior2nd', 'Exterior1st', 'Electrical', 'TotalBsmtSF',
    'BsmtUnfSF', 'BsmtFinSF2', 'KitchenQual', 'GarageArea', 'GarageCars', 'SaleType'
 ]

valid_error_cols = [c for c in error_missing_cols if c in train_df.columns]
dropped_error_cols = [c for c in error_missing_cols if c not in train_df.columns]

print(f'Colonnes a verifier pour NA erreurs : {valid_error_cols}')
if dropped_error_cols:
    print(f'Colonnes ignorees (absentes de train.csv): {dropped_error_cols}')

n_before_drop = len(train_df)
train_df = train_df.dropna(subset=valid_error_cols).reset_index(drop=True)
n_removed = n_before_drop - len(train_df)
print(f'Lignes supprimees sur train_df (NA erreurs): {n_removed}')

ntrain = len(train_df)
y_train_log = np.log1p(train_df['SalePrice'].copy())
test_ids = test_df['Id'].copy()
all_data = pd.concat([train_df.drop(columns=['SalePrice']), test_df], axis=0).reset_index(drop=True)
all_data = all_data.drop(columns=['Id'])

print(f'Shape all_data : {all_data.shape}')

DONNÉES MANQUANTES (34 colonnes)
   PoolQC               : 2907 ( 99.7%)
   MiscFeature          : 2810 ( 96.4%)
   Alley                : 2717 ( 93.2%)
   Fence                : 2345 ( 80.4%)
   MasVnrType           : 1765 ( 60.5%)
   FireplaceQu          : 1420 ( 48.7%)
   LotFrontage          :  486 ( 16.7%)
   GarageQual           :  159 (  5.5%)
   GarageYrBlt          :  159 (  5.5%)
   GarageCond           :  159 (  5.5%)
   GarageFinish         :  159 (  5.5%)
   GarageType           :  157 (  5.4%)
   BsmtExposure         :   82 (  2.8%)
   BsmtCond             :   82 (  2.8%)
   BsmtQual             :   81 (  2.8%)
   BsmtFinType2         :   80 (  2.7%)
   BsmtFinType1         :   79 (  2.7%)
   MasVnrArea           :   23 (  0.8%)
   MSZoning             :    4 (  0.1%)
   BsmtFullBath         :    2 (  0.1%)
   Functional           :    2 (  0.1%)
   BsmtHalfBath         :    2 (  0.1%)
   Utilities            :    2 (  0.1%)
   BsmtFinSF1           :    1 (  0.0%)
   Exte

## 3. Preprocessing de Base

**Objectif** : Traiter les valeurs manquantes structurelles.

**Actions** :
- Imputation par 'None' ou '0' selon le sens métier.
- Création de variables synthétiques globales (`TotalSF`).

In [31]:
# 2. PREPROCESSING DE BASE
def apply_base_preprocessing(data):
    df = data.copy()

    cols_none = ['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                 'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish',
                 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType']
    for col in cols_none:
        if col in df.columns:
            df[col] = df[col].fillna('None')

    cols_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2',
                 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
    for col in cols_zero:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    if 'LotFrontage' in df.columns :
        df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

    # Imputation par le mode pour les vrais manquants
    cols_mode = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType', 'Functional', 'Utilities']
    for col in cols_mode:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])

    # Choix volontaire : garder une variable synthétique de surface plutôt que plusieurs doublons exacts
    if {'TotalBsmtSF', '1stFlrSF', '2ndFlrSF'}.issubset(df.columns):
        df['TotalSF'] = df['TotalBsmtSF'].fillna(0) + df['1stFlrSF'].fillna(0) + df['2ndFlrSF'].fillna(0)
    if {'TotalBsmtSF', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF'}.issubset(df.columns):
        basement_total = df['TotalBsmtSF'].replace(0, np.nan)
        df['BasementFinishedShare'] = ((df['BsmtFinSF1'] + df['BsmtFinSF2']) / basement_total).fillna(0)
        df['BasementUnfinishedShare'] = (df['BsmtUnfSF'] / basement_total).fillna(0)

    if {'TotalSF', '2ndFlrSF'}.issubset(df.columns):
        total_sf = df['TotalSF'].replace(0, np.nan)
        df['SecondFloorShare'] = (df['2ndFlrSF'] / total_sf).fillna(0)

    if {'YrSold', 'YearBuilt'}.issubset(df.columns):
        df['AgeAtSale'] = df['YrSold'] - df['YearBuilt']
    if {'YrSold', 'YearRemodAdd'}.issubset(df.columns):
        df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']

    cols_to_drop = ['GarageArea', 'TotRmsAbvGrd', 'GarageYrBlt', 'YearBuilt', 'YearRemodAdd', 'YrSold',
                    'TotalBsmtSF', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', '1stFlrSF', '2ndFlrSF']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    return df

all_data_clean = apply_base_preprocessing(all_data)
X_train_clean = all_data_clean.iloc[:ntrain].copy()
X_test_clean = all_data_clean.iloc[ntrain:].copy()

print(f"Shape X_train_clean : {X_train_clean.shape}")
print(f"Shape X_test_clean  : {X_test_clean.shape}")

Shape X_train_clean : (1455, 73)
Shape X_test_clean  : (1459, 73)


## 4. Ingénierie des Catégories (Feature Engineering Avancé)

**Objectif** : Transformer les variables catégorielles complexes.

**Actions** :
- Encodage ordinal pour les échelles de qualité (Ex, Gd, TA, Fa, Po).
- Regroupement des catégories rares.
- Détection de colinéarité.

In [32]:
# 4. PIPELINE LASSO AVEC ENCODAGE CATEGORIEL
class AdvancedCategoricalEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, smoothing=10.0):
        self.smoothing = smoothing
        self.qual_mapping = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
        self.ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                             'HeatingQC', 'KitchenQual', 'FireplaceQu',
                             'GarageQual', 'GarageCond', 'PoolQC']

    def fit(self, X, y):
        X_copy = X.copy()
        for col in self.ordinal_cols:
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(self.qual_mapping).fillna(0)

        self.nominal_cols_ = X_copy.select_dtypes(include=['object', 'string']).columns.tolist()

        for col in self.nominal_cols_:
            X_copy[col] = X_copy[col].astype(str).fillna('Missing')

        self.te_ = TargetEncoder(cols=self.nominal_cols_, smoothing=self.smoothing)
        self.te_.fit(X_copy[self.nominal_cols_], y)
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in self.ordinal_cols:
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(self.qual_mapping).fillna(0)

        if self.nominal_cols_:
            for col in self.nominal_cols_:
                if col in X_copy.columns:
                    X_copy[col] = X_copy[col].astype(str).fillna('Missing')
            X_copy[self.nominal_cols_] = self.te_.transform(X_copy[self.nominal_cols_])

        return X_copy

numeric_features = X_train_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_clean.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('encoder', AdvancedCategoricalEngineer(smoothing=10.0))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])



## 5. Modélisation et Évaluation (Lasso)

**Objectif** : Entraîner le modèle et évaluer ses performances.

**Actions** :
- Mise en place du `Pipeline` avec preprocessing et `LassoCV`.
- Validation croisée (5-fold) pour obtenir le RMSE Log.
- Utilisation de `cross_val_predict` pour générer les prédictions OOF et calculer l'écart médian.
- Chronométrage de l'entraînement.

In [33]:
from sklearn.model_selection import cross_val_predict

# Début du chronomètre IT
start_time = time.perf_counter()

categorical_transformer = Pipeline(steps=[
    ('encoder', AdvancedCategoricalEngineer(smoothing=10.0))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

lasso_model = LassoCV(
    alphas=np.logspace(-6, 1, 1000),
    cv=5,
    random_state=42,
    max_iter=10000,
    tol=1e-4
)

pipeline_optimized = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', lasso_model)
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 1. Calcul du RMSE par validation croisée (Score mathématique)
cv_scores = cross_val_score(
    pipeline_optimized,
    X_train_clean,
    y_train_log,
    scoring='neg_root_mean_squared_error',
    cv=kf
)
rmse_cv_scores = np.abs(cv_scores)
rmse_log = rmse_cv_scores.mean()

# 2. Prédictions Out-Of-Fold pour l'Écart Médian (Score métier)
oof_preds_log = cross_val_predict(pipeline_optimized, X_train_clean, y_train_log, cv=kf)
oof_preds_dollars = np.expm1(oof_preds_log)
y_train_dollars = np.expm1(y_train_log)

ecart_pct = np.abs(oof_preds_dollars - y_train_dollars) / y_train_dollars * 100
ecart_median = np.median(ecart_pct)
ecart_moyen_dollars = np.mean(np.abs(oof_preds_dollars - y_train_dollars))

# 3. Entraînement final sur tout le jeu de données
pipeline_optimized.fit(X_train_clean, y_train_log)
alpha_opt = pipeline_optimized.named_steps['model'].alpha_

# Fin du chronomètre IT
end_time = time.perf_counter()
train_time = end_time - start_time

print(f"Alpha optimal trouvé : {alpha_opt:.6f}")

Alpha optimal trouvé : 0.000237


## 6. Prédictions Finales et Affichage

**Objectif** : Générer le fichier de soumission et afficher le résumé complet.

**Actions** :
- Prédictions sur le jeu de test.
- Affichage des statistiques de prédiction.
- Impression du résumé final avec les 3 métriques clés.

In [34]:
y_pred_log_opt = pipeline_optimized.predict(X_test_clean)
y_pred_dollars_opt = np.expm1(y_pred_log_opt)

# Soumission
submission = pd.DataFrame({
    'Id': test_ids.values,
    'SalePrice': y_pred_dollars_opt
})
submission.to_csv('M1_Lasso_V8_Final.csv', index=False)

n_features_total = pipeline_optimized.named_steps['model'].coef_.shape[0]
n_features_retained = int(np.sum(pipeline_optimized.named_steps['model'].coef_ != 0))

# Affichage des statistiques
print("=" * 50)
print("  STATISTIQUES DES PRÉDICTIONS")
print("=" * 50)
print(f"  Min      : ${y_pred_dollars_opt.min():,.0f}")
print(f"  Max      : ${y_pred_dollars_opt.max():,.0f}")
print(f"  Médiane  : ${np.median(y_pred_dollars_opt):,.0f}")
print(f"  Moyenne  : ${y_pred_dollars_opt.mean():,.0f}")
print(f"  Features : {n_features_retained}/{n_features_total} retenues")

# Affichage des trois critères
print("\n" + "=" * 50)
print("  ÉVALUATION FINALE — MODÈLE M1 LASSO")
print("=" * 50)
print(f"  Mathématique (RMSE Log) : {rmse_log:.5f}")
print(f"  IT (Temps entraînement) : {train_time:.2f} secondes")
print(f"  Métier (Écart médian)   : {ecart_median:.2f} %")
print(f"  Métier (Écart moyen $)  : ${ecart_moyen_dollars:,.0f}")
print("=" * 50)

  STATISTIQUES DES PRÉDICTIONS
  Min      : $53,311
  Max      : $1,382,997
  Médiane  : $157,409
  Moyenne  : $178,544
  Features : 55/73 retenues

  ÉVALUATION FINALE — MODÈLE M1 LASSO
  Mathématique (RMSE Log) : 0.11874
  IT (Temps entraînement) : 8.41 secondes
  Métier (Écart médian)   : 6.20 %
  Métier (Écart moyen $)  : $14,679
